# Naive RAG - "Os Sertões" de Euclides da Cunha

Implementação da abordagem mais simples de RAG (Retrieval-Augmented Generation):

1. **Carregamento** do PDF
2. **Chunking** com `RecursiveCharacterTextSplitter` (tamanho fixo)
3. **Embedding** com `sentence-transformers` (modelo multilingual)
4. **Armazenamento** em ChromaDB
5. **Retrieval** por similaridade cosseno (top-k)
6. **Geração** via OpenAI GPT

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Defina sua chave aqui ou em um arquivo .env
# os.environ["OPENAI_API_KEY"] = "sk-..."

assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no .env ou como variável de ambiente"

## 1. Carregamento do PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "os-sertoes.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Total de páginas carregadas: {len(pages)}")
print(f"Exemplo (página 10):\n{pages[10].page_content[:500]}")

## 2. Chunking (Divisão em Chunks)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Total de chunks gerados: {len(chunks)}")
print(f"\nExemplo de chunk:\n{chunks[50].page_content[:300]}")
print(f"\nMetadados: {chunks[50].metadata}")

## 3. Embeddings e Vector Store (ChromaDB)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

CHROMA_DIR = "./chroma_naive"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="os_sertoes_naive"
)

print(f"Vector store criado com {vectorstore._collection.count()} documentos")

## 4. Configuração do Retriever e LLM

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=2000
)

PROMPT_TEMPLATE = """
Você é um especialista em literatura brasileira, especificamente na obra "Os Sertões" de Euclides da Cunha.
Responda a pergunta com base EXCLUSIVAMENTE nos trechos do livro fornecidos abaixo.
Se a informação não estiver nos trechos, diga que não encontrou a informação no contexto fornecido.
Cite trechos relevantes quando possível.

Contexto:
{context}

Pergunta: {question}

Resposta detalhada:
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Página {d.metadata.get('page', '?')}]\n{d.page_content}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Pipeline Naive RAG configurado com sucesso!")

## 5. Execução das Perguntas

In [ ]:
PERGUNTAS = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em 'Os Sertões'? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

In [ ]:
respostas_naive = {}

for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"{'='*80}")
    print(f"PERGUNTA {i}: {pergunta}")
    print(f"{'='*80}")
    
    resposta = rag_chain.invoke(pergunta)
    respostas_naive[i] = resposta
    
    print(f"\nRESPOSTA:\n{resposta}")
    print()

## 6. Análise dos Chunks Recuperados

Vamos inspecionar quais trechos foram recuperados para cada pergunta.

In [ ]:
for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"\n{'='*80}")
    print(f"PERGUNTA {i}: {pergunta[:80]}...")
    print(f"{'='*80}")
    
    docs = retriever.invoke(pergunta)
    
    for j, doc in enumerate(docs, 1):
        print(f"\n--- Chunk {j} (Página {doc.metadata.get('page', '?')}) ---")
        print(doc.page_content[:200] + "...")